# Week 1 — Data Ingestion & Verification
**Project:** Flight Delay Analytics
**Goal:** Confirm the flight delay dataset loaded correctly into SQLite and is ready for SQL analysis (Week 2).

**Data source:** Kaggle — `flight_data_2024` (BTS on-time performance data, ~7M rows, 2024)

**Note on method:** The original plan was to load data via `src/ingest.py`. The Kaggle CSV's column names (e.g. `op_unique_carrier`, `arr_delay`) didn't match the raw-BTS column names the script expected, so the CSV was instead imported directly into SQLite using DB Browser's **File → Import → Table from CSV** feature. This notebook verifies that import and explores the resulting schema.

## 1. Connect to the database

In [1]:
import pandas as pd
import sqlite3
import os

DB_PATH = os.path.join("..", "data", "flights.db")
TABLE_NAME = "flights_raw"  # matches the name DB Browser gave the imported table

conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")
print(f"Database file size: {os.path.getsize(DB_PATH) / 1e6:.1f} MB")

Connected to: ..\data\flights.db
Database file size: 958.6 MB


## 2. Confirm row count

In [2]:
row_count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {TABLE_NAME}", conn).iloc[0, 0]
print(f"Total rows in {TABLE_NAME}: {row_count:,}")

Total rows in flights_raw: 7,079,081


## 3. Inspect the schema

In [3]:
schema = pd.read_sql(f"PRAGMA table_info({TABLE_NAME})", conn)
print(f"Total columns: {len(schema)}\n")
schema[["cid", "name", "type"]]

Total columns: 35



,cid,name,type
0,0,year,INTEGER
1,1,month,INTEGER
2,2,day_of_month,INTEGER
3,3,day_of_week,INTEGER
4,4,fl_date,TEXT
5,5,op_unique_carrier,TEXT
6,6,op_carrier_fl_num,REAL
7,7,origin,TEXT
8,8,origin_city_name,TEXT
9,9,origin_state_nm,TEXT


## 4. Preview the data

In [4]:
preview = pd.read_sql(f"SELECT * FROM {TABLE_NAME} LIMIT 10", conn)
preview

,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,...,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,1,1,1,2024-01-01,9E,4814.0,JFK,"New York, NY",New York,...,0,136.0,122.0,84.0,509.0,0,0,0,0,0
1,2024,1,1,1,2024-01-01,9E,4815.0,MSP,"Minneapolis, MN",Minnesota,...,0,130.0,114.0,88.0,622.0,0,0,0,0,0
2,2024,1,1,1,2024-01-01,9E,4817.0,JFK,"New York, NY",New York,...,0,106.0,90.0,61.0,288.0,0,0,0,0,0
3,2024,1,1,1,2024-01-01,9E,4817.0,RIC,"Richmond, VA",Virginia,...,0,111.0,76.0,51.0,288.0,0,0,0,0,0
4,2024,1,1,1,2024-01-01,9E,4818.0,DTW,"Detroit, MI",Michigan,...,0,79.0,70.0,45.0,237.0,0,0,0,0,0
5,2024,1,1,1,2024-01-01,9E,4822.0,JAX,"Jacksonville, FL",Florida,...,0,137.0,120.0,102.0,833.0,0,0,0,0,0
6,2024,1,1,1,2024-01-01,9E,4822.0,LGA,"New York, NY",New York,...,0,169.0,164.0,125.0,833.0,0,0,0,0,0
7,2024,1,1,1,2024-01-01,9E,4823.0,CHS,"Charleston, SC",South Carolina,...,0,118.0,99.0,86.0,641.0,0,0,0,0,0
8,2024,1,1,1,2024-01-01,9E,4823.0,LGA,"New York, NY",New York,...,0,149.0,123.0,101.0,641.0,0,0,0,0,0
9,2024,1,1,1,2024-01-01,9E,4828.0,ITH,"Ithaca/Cortland, NY",New York,...,0,79.0,67.0,43.0,189.0,0,0,0,0,0


## 5. Quick sanity checks
A few numbers to confirm the data makes sense before building SQL queries on top of it.

In [5]:
# Date range covered
date_range = pd.read_sql(f"""
SELECT MIN(fl_date) AS earliest, MAX(fl_date) AS latest
FROM {TABLE_NAME}
""", conn)
date_range

,earliest,latest
0,2024-01-01,2024-12-31


In [6]:
# Number of distinct carriers and airports
summary = pd.read_sql(f"""
SELECT
    COUNT(DISTINCT op_unique_carrier) AS num_carriers,
    COUNT(DISTINCT origin)            AS num_origin_airports,
    COUNT(DISTINCT dest)              AS num_dest_airports
FROM {TABLE_NAME}
""", conn)
summary

,num_carriers,num_origin_airports,num_dest_airports
0,15,348,348


## 6. Close the connection

## Next step
Week 2: write the first 5 analytical SQL queries (delay by carrier, by airport, by month, on-time rate trend, top delay causes).

In [7]:
conn.close()
print("Connection closed. Week 1 verified — ready for Week 2 SQL queries.")

Connection closed. Week 1 verified — ready for Week 2 SQL queries.
